وقتی داده ها فیلتر شدند این برنامه نمودار تولید بر اساس دما را برای تمام واحدهای تولیدی رسم کرده و داده های مربوط به آخرین سری تولید و داده های انتخاب شده توسط فیلتر پنجم را نیز مشخص میکند. 

In [1]:
import os
import sys
import plotly.graph_objects as go

import matplotlib.pyplot as plt

current_dir = os.getcwd()
project_root = current_dir[:current_dir.find("src") - 1]
sys.path.insert(0, project_root)
from src.models.filter_data.filter_data import *
from src.models.filter_data.feature_adder import *

In [2]:
l_min = 4
max_diff = 3
c_thresh = 0.9

csv_read_path = os.path.join(project_root, "data", "processed", "semi_processed.csv")

df = pd.read_csv(csv_read_path, encoding='utf-8')
df_c = df

In [15]:
csv_read_path = os.path.join(project_root, "data", "interim", "factors.csv")
df_factors = pd.read_csv(csv_read_path)
df_factors

coefs = {}
grouped = df_factors.groupby(['PowerPlantCode', 'PowerPlantName', "UnitCode"])

for (pp_code, pp_name, unit_code), g in grouped:
    uniques = g[["a1IndexGas", "b1IndexGas"]].drop_duplicates()
    coefs[(pp_name, unit_code)] = []
    for row in uniques.itertuples(index=False):
        coefs[(pp_name, unit_code)].append((row.a1IndexGas, row.b1IndexGas))

In [47]:
import plotly.express as px


def show(df_m1, save=False, param=None, ass=""):
    fig = px.scatter(df_m1, x="datetime", y='generation', color='is_good_peak',
                     title='Generation over Time by Batch Interval',
                     labels={'generation': 'Generation', 'datetime': 'Time'},
                     hover_data=['datetime', 'generation', 'temp_sens', 'interval_id'])

    if save:
        fig.write_html(
            f"{project_root}/src/visualization/unit_figs/filter5{ass}/{param['name']}-{param['code']}_l.html")
    else:
        fig.show()

    fig = px.scatter(df_m1, x="temp_sens", y='generation', color='is_good_peak',
                     title='Generation over Time by Batch Interval',
                     labels={'generation': 'Generation'},
                     hover_data=['datetime', 'generation', 'temp_sens'],size_max=1)
    
    fig.update_traces(
    marker=dict(
        size=4,           # اندازه ثابت
        sizemode='diameter',  # یا 'area'
        sizeref=1,        # برای مقیاس‌بندی
        opacity=0.7       # شفافیت
    )
)
    
    n = param["name"]
    c = param["code"]
    for i in range(len(coefs[(n, c)])):
        a, b = coefs[(n, c)][i]
        x_line = np.linspace(df_m1["temp_sens"].min(), df_m1["temp_sens"].max(), 100)
        y_line = a * x_line + b
        
        fig.add_trace(go.Scatter(
            x=x_line,
            y=y_line,
            mode="lines",
            name=f"y = {a}x + {b}",
            line=dict(dash="dash", width=2)
        ))
    
    if save:
        fig.write_html(
            f"{project_root}/src/visualization/unit_figs/filter5{ass}/{param['name']}-{param['code']}_s.html")
    else:
        fig.show()
        
    

In [37]:
power_plants = df_c[['name', 'code']].drop_duplicates()

for row in power_plants.itertuples():
    name_plot, code_plot = row.name, row.code
    ds_n_c_plot = Data_selector(Data_selector(df_c).select_peaks(goodness=2))
    df_n_c_plot = ds_n_c_plot.filter_name_code(name_plot, code_plot)
    show(df_n_c_plot, save=True, param={"name": name_plot, "code": code_plot})

In [48]:
name_plot, code_plot = "پرند", "G12"
ds_n_c_plot = Data_selector(Data_selector(df_c).select_peaks(goodness=0))
df_n_c_plot = ds_n_c_plot.filter_name_code(name_plot, code_plot)
show(df_n_c_plot, save=False, param={"name": name_plot, "code": code_plot})